# Similarity metrics

In [1]:
import math
import random
import warnings
warnings.filterwarnings("ignore")

from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import similaritymeasures
from sdtw.soft_dtw import SoftDTW

## Data loading

In [2]:
df = pd.read_csv("hsa_state_new.csv", parse_dates=["time_value"])
df = df.dropna(subset=["state", "hsa_value", "state_value"])
df = df.sort_values(["hsa_id", "time_value"]).reset_index(drop=True)

print(f"Rows       : {len(df):,}")
print(f"States     : {df['state'].nunique()}")
print(f"HSAs       : {df['hsa_id'].nunique()}")
print(f"Date range : {df['time_value'].min().date()} to {df['time_value'].max().date()}")
df.head()

Rows       : 125,839
States     : 38
HSAs       : 671
Date range : 2022-09-25 to 2026-05-17


,time_value,hsa_id,state,hsa_value,state_value,hsa_pop,state_pop,pop_ratio
0,2022-09-25,1,Maryland,0.00,0.26,96475.0,6170738.0,0.015634
1,2022-10-02,1,Maryland,0.00,0.41,96475.0,6170738.0,0.015634
2,2022-10-09,1,Maryland,0.00,0.93,96475.0,6170738.0,0.015634
3,2022-10-16,1,Maryland,0.19,1.80,96475.0,6170738.0,0.015634
4,2022-10-23,1,Maryland,0.70,3.74,96475.0,6170738.0,0.015634


In [3]:
hsa_list = df["hsa_id"].values
chosen_id = random.choice(hsa_list)

hsa_display = df[df['hsa_id'] == chosen_id]

dates = hsa_display["time_value"].values
hsa_curve = hsa_display["hsa_value"].values
state_curve = hsa_display["state_value"].values

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=hsa_curve, name='HSA'))
fig.add_trace(go.Scatter(x=dates, y=state_curve, name='State'))
fig.update_layout(title=f"HSA {chosen_id} ({hsa_display['state'].iloc[0]})")
fig.update_xaxes(dtick="M1")
fig.show()

## Helper functions

In [4]:
def make_curve(t_days, values):
    curve = np.zeros((len(t_days), 2))
    curve[:, 0] = t_days
    curve[:, 1] = values
    return curve

print("Helpers ready")

Helpers ready


## HSA analysis
### Window specification

In [5]:
records = []

groups = list(df.groupby("hsa_id"))
n_total = len(groups)

for idx, (hsa_id, grp) in enumerate(groups):

    if idx % 100 == 0:
        print(f"  {idx}/{n_total} HSAs processed...")

    grp = grp.sort_values("time_value")
    if len(grp) < 4:
        continue

    t_days     = (grp["time_value"] - grp["time_value"].min()).dt.days.values.astype(float)
    state_vals = grp["state_value"].values.astype(float)
    hsa_vals   = grp["hsa_value"].values.astype(float)

    # exp_data = ground truth (state), num_data = local signal (hsa)
    exp_data = make_curve(t_days, state_vals)
    num_data = make_curve(t_days, hsa_vals)

    area      = similaritymeasures.area_between_two_curves(exp_data, num_data)
    rmse       = math.sqrt(similaritymeasures.mse(exp_data, num_data))

    path = warp_path_from_matrix(dtw_matrix)
    lags = lag_metrics(path)

    records.append({
        "hsa_id"             : hsa_id,
        "state"              : grp["state"].iloc[0],
        "n_obs"              : len(grp),
        "pop_ratio"          : grp["pop_ratio"].iloc[0],
        "abc"                : area,
        "dtw"                : dtw_dist,
        "rmse"                : rmse,
        "timing_component"   : timing_component,
        "magnitude_component": dtw_dist,
        "timing_share"       : timing_share,
        **lags,
    })

hsa_df = pd.DataFrame(records)
print(f"Done — {len(hsa_df)} HSAs")
hsa_df.round(4).head(10)

  0/671 HSAs processed...


NameError: name 'warp_path_from_matrix' is not defined

# SoftDTW



# SimilarityMetrics


In [4]:
hsa_df.to_csv("hsa_metrics.csv", index=False)
print("Saved: hsa_metrics.csv")

NameError: name 'hsa_df' is not defined


# Analysis by HSA



# Analysis by State